# LEAVE ONE VARIANT OUT 
This script submits the commands for the leave-one-variant-out (LOVO) analysis incorporating the deprivation index. It performs Step 2 of the SAIGE-GENE+ association tests for each variant set, where each set has exactly one variant removed from the full collection. The script iterates over all annotation files, with each file representing a reduced variant set missing one specific variant. By looping through these files, the pipeline re-runs Step 2 for every leave-one-out subset, allowing for evaluation of each variant’s individual contribution to the overall gene-based association signal and identifying variants that disproportionately influence the test statistic.

In [ ]:
# Imports the 'os' module, which allows interaction with the operating system 
import os 

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' 
my_bucket


# COMMAND 

In [ ]:
%%writefile lovo_high_confidence_deprivation_index_analysis.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Define path to lovo annotation file 
annotation_list="${vat}/LOVO-annotation-chr${chrom}.txt"

# Loop through each population 
for pop in EUR; do
  
  # Loop through each phenotype  
  for pheno in aml_phenotype; do
   
    while IFS= read -r variant; do
    
        echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom} | variant: ${variant}"
        
        # Loop through each lovo annotation file  
        for annotation_table in ${vat}/LOVO-${variant}-annotation-*.txt; do
        
        echo "Processing annotation table: ${annotation_table}"
        
        step2_SPAtests.R \
            --bedFile="${plink_bed}" \
            --bimFile="${plink_bim}" \
            --famFile="${plink_fam}" \
            --SAIGEOutputFile="${output}/${pop}_${pheno}_lovo_high_confidence_${variant}_deprivation_index" \
            --LOCO=FALSE \
            --AlleleOrder=alt-first \
            --minMAF=0 \
            --minMAC=0.5 \
            --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_deprivation_index_null_glmm.rda" \
            --varianceRatioFile="${null_glmm}/${pop}_${pheno}_deprivation_index_null_glmm.varianceRatio.txt" \
            --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
            --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
            --groupFile="${annotation_table}" \
            --annotation_in_groupTest=AlphaMissense-0.564,LOFTEE-HC:SpliceAI-0.8,LOFTEE-HC:SpliceAI-0.8:AlphaMissense-0.564 \
            --maxMAF_in_groupTest=0.0001,0.001,0.01 \
            --is_output_markerList_in_groupTest=TRUE \
            --is_fastTest=TRUE

        done
    done < "${annotation_list}"

    done
done


In [ ]:
# Copy script in workspace bucket 
!gsutil cp lovo_high_confidence_deprivation_index_analysis.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/lovo_high_confidence_deprivation_index_analysis.sh

# SUBMISSION

In [ ]:
%%bash --out saige_gene_step02

# Loop through the each chromosome 
for chromo in 2 4 15 17; do

# Source the dsub setup script
source ~/aou_dsub.bash 

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_LOVO_annotation/EUR_LOVO_annotation_chr${chromo}/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/LOVO_analysis/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/lovo_high_confidence_deprivation_index_analysis.sh"

done